## 그룹화 및 Aggregation

In [32]:
import pandas as pd

# 예시 데이터프레임: 팀 별 점수
data = {'Team': ['Red', 'Blue', 'Red', 'Blue'],
        'Score': [5, 3, 8, 7],
        'Passion' : [95, 88, 63, 98]}
df = pd.DataFrame(data)
print("원본 데이터프레임:\n", df)

원본 데이터프레임:
    Team  Score  Passion
0   Red      5       95
1  Blue      3       88
2   Red      8       63
3  Blue      7       98


In [33]:
# Team별 Score 합계
grouped_sum = df.groupby('Team')['Score'].sum()
print("팀별 점수 합계:\n", grouped_sum)
# 팀별로 그룹화 한 후 sum()을 이용하여 점수 합산 -> 시리즈임

팀별 점수 합계:
 Team
Blue    10
Red     13
Name: Score, dtype: int64


In [34]:
grouped_mean = df.groupby('Team')['Score'].mean()
print("팀별 점수 평균:\n", grouped_mean)
# 위와 마찬가지로 팀별로 그룹화 하여 평균을 구할 수 있음.

팀별 점수 평균:
 Team
Blue    5.0
Red     6.5
Name: Score, dtype: float64


In [35]:
multi_agg = df.groupby('Team').agg(
    Score_sum=('Score', 'sum'),
    Score_mean=('Score', 'mean'),
    Passion_sum=('Passion', 'sum'),
    Passion_mean=('Passion', 'mean')
)
print("팀별 점수/열정 평균과 합계:\n", multi_agg)

# .agg() 메서드를 이용하여 여러 함수 한꺼번에 적용
# 팀별로 점수 합계와 평균, 열정 합계와 평균 구한다.

팀별 점수/열정 평균과 합계:
       Score_sum  Score_mean  Passion_sum  Passion_mean
Team                                                  
Blue         10         5.0          186          93.0
Red          13         6.5          158          79.0


In [36]:
# 타이타닉 데이터로 실습해보기
import pandas as pd
titanic_df = pd.read_csv('train.csv')
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [37]:
# 객실등급(Pclass)에 따른 평균 요금

class_fare_mean = titanic_df.groupby('Pclass')['Fare'].mean()
class_fare_mean

Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64

In [38]:
# 성별에 따른 생존율
survival_rate_by_gender = titanic_df.groupby('Sex')['Survived'].mean()
survival_rate_by_gender

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [39]:
# 객실등급과 성별 두 가지 기준으로 나눈뒤 생존률 계산

pclass_sex_survived = titanic_df.groupby(['Pclass','Sex'])['Survived'].mean()
pclass_sex_survived

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

In [40]:
# 각 탑승항구 별 생존률

survived_by_Embarked = titanic_df.groupby('Embarked')['Survived'].mean()
survived_by_Embarked

Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64

In [41]:
# 객실등급별 최고령 승객 나이

max_age_by_pclass = titanic_df.groupby('Pclass')['Age'].max()
max_age_by_pclass

Pclass
1    80.0
2    70.0
3    74.0
Name: Age, dtype: float64

## Transform? > 그룹 계산을 원본 길이 유지한 채로 붙이기
- transform 이란 각 그룹에서 계산한 값을 원본 인덱스에 맞춰 되돌려준다는 것

In [42]:
# 등급별 평균 요금
class_mean_fare = titanic_df.groupby("Pclass")["Fare"].mean()

# 등급별 평균 요금을 각 행마다(각 행의 등급마다) 보여줌
class_mean_fare_transform = titanic_df.groupby("Pclass")["Fare"].transform("mean")

In [43]:
# 등급별 평균 요금
class_mean_fare = titanic_df.groupby("Pclass")["Fare"].transform("mean")

# 평균 대비 비율
titanic_df["Fare_VS_ClassMean"] = titanic_df["Fare"] / class_mean_fare
titanic_df[["Pclass", "Fare", "Fare_VS_ClassMean"]].head()

# 각 승객의 요금을 같은 등급 평균 대비 얼마나 비싼지 알고 싶다면?
# transform을 이용해 각 등급별 평균 금액을 각 행마다 붙여주고
# 평균 대비 비율을 계산해 새로운 컬럼에 추가한다.

,Pclass,Fare,Fare_VS_ClassMean
0,3,7.2500,0.530143
1,1,71.2833,0.847051
2,3,7.9250,0.579501
3,1,53.1000,0.630981
4,3,8.0500,0.588642


In [44]:
age_group_median = titanic_df.groupby(["Pclass", "Sex"])["Age"].transform("median")
titanic_df["Age"] = titanic_df["Age"].fillna(age_group_median)

## unstack: MultiIndex 결과를 표로 펼치기
- unstack은 멀티인덱스를 가진 시리즈를 데이터프레임(표)로 피벗처럼 펼치는 기능
- 기본적으로 마지막 인덱스 레벨을 열로 올린다 (컬럼).

In [45]:
surv_table = (
    titanic_df.groupby(["Pclass","Sex"])["Survived"].mean()
    .unstack()
)
surv_table
# groupby의 마지막 인덱스인 Sex를 컬럼으로 올린것을 볼 수 있다.

Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


- fill_value= : 펼치면서 생긴 NaN을 원하는 값으로 대체(default: None)
- level= : 어떤 레벨을 펼칠지 지정 가능(default: 마지막 인덱스 레벨)

In [46]:
surv_table2 = surv_table.unstack(fill_value=0)
surv_table2

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
dtype: float64

In [47]:
# Pclass × Embarked 생존률 표(Dataframe)를 만들고, 빈 조합은 0으로 채워라
titanic_df.groupby(['Pclass', 'Embarked'])['Survived'].mean().unstack(fill_value=0)

Embarked,C,Q,S
Pclass,,,
1,0.694118,0.500000,0.582677
2,0.529412,0.666667,0.463415
3,0.378788,0.375000,0.189802


## reshape: melt / pivot / pivot_table

**reshape : 표 모양 바꾸기**

방금 배운 unstack/stack 도 reshape를 위한 어러 도구 중 하나

## wide vs long 개념

**wide(와이드) → 보고서용**

- 변수(카테고리)가 열로 퍼져있는 형태
- 예: 월별 매출이 Jan/Feb/Mar… 컬럼으로 있음

**long(롱=tidy: 정리정돈이 잘 된 데이터) → 시각화/분석용**

- 하나의 관측값이 한 행에 가깝게 정리된 형태
- 예: (사람, 월, 매출)처럼 **카테고리(월)가 행으로 내려옴**

In [48]:
sales_wide = pd.DataFrame({
    "Name": ["Kim", "Lee"],
    "Jan": [100, 200],
    "Feb": [120, 210],
    "Mar": [130, 190],
})

sales_long = sales_wide.melt(
    id_vars=["Name"],
    var_name="Month",
    value_name="Sales"
)

# id_vars는 그대로 유지할 식별자 컬럼,
# 나머지(value_vars)는 행으로 내림.
# melt: wide -> long

display(sales_wide)
display(sales_long)

,Name,Jan,Feb,Mar
0,Kim,100,120,130
1,Lee,200,210,190


,Name,Month,Sales
0,Kim,Jan,100
1,Lee,Jan,200
2,Kim,Feb,120
3,Lee,Feb,210
4,Kim,Mar,130
5,Lee,Mar,190


In [49]:
sales_wide_back = sales_long.pivot(index="Name", columns="Month", values="Sales")
sales_wide_back

# pivot: long -> wide
# index,columns 조합이 유일해야 한다. 유일하지 않으면 에러남.

Month,Feb,Jan,Mar
Name,,,
Kim,120,100,130
Lee,210,200,190


In [50]:
sales_long_dup = pd.DataFrame({
    "Name": ["Kim", "Kim", "Lee"],
    "Month": ["Jan", "Jan", "Jan"],
    "Sales": [100, 50, 200]
})

# pivot은 (Name, Month)가 중복이라 곤란할 수 있음
# pivot_table은 aggfunc로 처리 가능 (집계)

pt = sales_long_dup.pivot_table(
    index="Name",
    columns="Month",
    values="Sales",
    aggfunc="sum",      # 합계로 집계
    fill_value=0
)
pt

# pivot()은 데이터 구조를 변경하고 피벗팅할 때 사용되며,
# pivot_table()은 데이터를 요약하고 집계할 때 사용됩니다.

Month,Jan
Name,
Kim,150
Lee,200


In [51]:
# sales_long_dup에서 월별 평균 매출 피벗을 만들어라. 빈 값은 0으로.
pt_mean = sales_long_dup.pivot_table(
    index = "Name",
    columns = "Month",
    values = "Sales",
    aggfunc = "mean",
    fill_value = 0
)
pt_mean

Month,Jan
Name,
Kim,75.0
Lee,200.0


In [52]:
# Pclass × Sex 생존률 표를 만들어라. (pivot_table )
titanic_pivot = titanic_df.pivot_table(
    index = "Pclass",
    columns = "Sex",
    values = "Survived",
    aggfunc = "mean"
)
titanic_pivot

Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


In [53]:
# 위 피벗 표를 다시 long으로 바꿔서 그래프용 데이터 만들어라. (melt - 식별자:Pcalss)
titanic_pivot_melt = titanic_pivot.reset_index().melt(
    id_vars= "Pclass",
    var_name = "Sex",
    value_name = "Surv_rate"
)
titanic_pivot_melt
# pivot → index 생김 → reset_index로 컬럼 복귀 → melt 가능

,Pclass,Sex,Surv_rate
0,1,female,0.968085
1,2,female,0.921053
2,3,female,0.500000
3,1,male,0.368852
4,2,male,0.157407
5,3,male,0.135447


### +그래서 어떤 상황에서 뭐 어떻게 하라고?

- **내 데이터가 이미 MultiIndex 결과물이다** (예: `groupby`하고 나니 인덱스가 2~3레벨)
    
    → `unstack()`으로 보고 싶은 축을 컬럼으로 올려서 표 만들기
    
    → 다시 길게 돌리고 싶으면 **`stack()`**
    
- **내 데이터가 컬럼이 너무 많고(가로로 넓고), 변수명이 컬럼에 박혀 있다** (예: `sales_2020`, `sales_2021` / `Jan, Feb, Mar…`)
    
    → **`melt()`(=unpivot)** 로 long(tidy) 만들기
    
- **내 데이터가 long(tidy)인데 매트릭스 형태로 보고 싶다**
    
    → 키 조합이 유일하면 **`pivot()`**
    
    → 유일하지 않으면 먼저 집계하거나 **`pivot_table()`**

## 문자열 처리

> 문자열 처리는 `Series.str`로 **열 전체**에 적용한다.
> 

- **문자열 데이터 처리**는 매우 중요합니다. 이름, 주소, 코드 등 **텍스트 형태 데이터**에서 필요한 정보를 추출하거나 형식을 변환하는 작업이 잦습니다.
- Pandas에서는 문자열 전용 메서드를 사용하기 위해 **`Series.str`** 속성을 제공합니다.
- `df['컬럼'].str.메서드()` 형식으로 사용하며, Python의 문자열 메서드들을 Series 전체에 적용할 수 있습니다.
- 자주 사용하는 문자열 처리 기능:
    - **대소문자 변환:** `.lower()`, `.upper()`, `.capitalize()`, `.title()` 등
    - **공백 제거:** `.strip()` (문자열 양 끝 공백 제거), `.lstrip()`, `.rstrip()`
    - **길이 계산:** `.len()` 각 문자열의 길이 반환
    - **검색/포함 여부:** `.contains("문자열")` 부분 문자열 포함 여부 (True/False 시리즈 반환), `.startswith()`, `.endswith()`
    - **치환:** `.replace("기존", "새로운")` 문자열 치환 (정규표현식 사용 가능)
    - **분할:** `.split("구분자")` 구분자를 기준으로 문자열을 나누어 리스트로 반환; 이후 `[index]`로 각 부분 추출 가능
    - 이 외에도 `.find()`, `.isdigit()`, `.join()` 등 다양한 메서드 활용 가능
- **주의:** `Series.str` 메서드는 **NaN (결측치)**가 있으면 작업이 불가능하므로, 먼저 결측치를 처리하거나 제거해야 함.

In [54]:
import pandas as pd

names = pd.Series(['Alice', ' Bob ', 'Charlie'])
print("원본 시리즈:\n", names)

원본 시리즈:
 0      Alice
1       Bob 
2    Charlie
dtype: object


In [55]:
print("모든 이름 소문자:\n", names.str.lower())
print("모든 이름 대문자:\n", names.str.upper())

모든 이름 소문자:
 0      alice
1       bob 
2    charlie
dtype: object
모든 이름 대문자:
 0      ALICE
1       BOB 
2    CHARLIE
dtype: object


In [56]:
print("양쪽 공백 제거:\n", names.str.strip())

양쪽 공백 제거:
 0      Alice
1        Bob
2    Charlie
dtype: object


In [57]:
print("이름 길이:\n", names.str.strip().str.len())

이름 길이:
 0    5
1    3
2    7
dtype: int64


In [58]:
print("li 포함 여부:\n", names.str.contains("li"))

li 포함 여부:
 0     True
1    False
2     True
dtype: bool


In [59]:
names = pd.DataFrame({
	"Name":['Alice', ' Bob ', 'Charlie'],
	"Age": [22, 25, 36],
	'introduce':['hello. Alice', 'hello. Bob', 'hello. Charlie']})

print(names['introduce'].str.split('.').str[1])

0       Alice
1         Bob
2     Charlie
Name: introduce, dtype: object


### **Titanic 데이터 실습**

- **예제 1: 승객 이름에서 성(last name) 추출**
    - Titanic 데이터의 **Name** 열에는 `"성, 호칭. 이름"` 형태로 이름이 적혀 있습니다 (예: **"Braund, Mr. Owen Harris"**).
    - 각 승객의 **성을 추출**하여 새로운 열(`LastName`)로 추가해보겠습니다.

In [60]:
titanic_df['LastName'] = titanic_df['Name'].str.split(',').str[0]
titanic_df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fare_VS_ClassMean,LastName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,0.530143,Braund


- **예제 2: 특정 문자열 포함 여부로 데이터 필터링**
    - **미혼 여성(Miss.) 승객**이 몇 명인지 확인해보겠습니다.
    - Name 열에 `"Miss."`라는 문자열이 포함되는지를 검사하여 True/False 불린 시리즈를 얻은 뒤, True인 데이터만 추려서 개수를 세어봅시다.

In [61]:
Miss_cnt = titanic_df['Name'].str.contains('Miss.')

len(titanic_df.loc[Miss_cnt])

182

- **예제 3: 문자열 치환 및 수정** – 이번에는 승객 이름에서 괄호를 없애보겠습니다.

In [62]:
titanic_df['Cleaned_name'] = titanic_df['Name'].str.replace('(', '').str.replace(')', '')
titanic_df[['Name','Cleaned_name']].head()

,Name,Cleaned_name
0,"Braund, Mr. Owen Harris","Braund, Mr. Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","Cumings, Mrs. John Bradley Florence Briggs Thayer"
2,"Heikkinen, Miss. Laina","Heikkinen, Miss. Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)","Futrelle, Mrs. Jacques Heath Lily May Peel"
4,"Allen, Mr. William Henry","Allen, Mr. William Henry"


1. **퀴즈 1:** Titanic 승객들의 이름에서 **호칭(Title)**을 추출해보세요. 예를 들어 "Braund, **Mr**. Owen Harris"에서 **"Mr"**, "Cumings, **Mrs**. John ..."에서 **"Mrs"**를 얻어야 합니다. 
    1. (힌트: 이름은 **콤마(,)**와 **마침표(.)**를 기준으로 "성, 호칭. 이름" 구성입니다. 먼저 콤마로 분리한 뒤, 두 번째 부분에서 마침표 이전까지를 가져오면 됩니다.)

In [86]:
titanic_df['Title'] = titanic_df['Name'].str.split(',').str[1].str.split('.').str[0]
titanic_df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fare_VS_ClassMean,LastName,Cleaned_name,Title,BoardTime,Embarked_low
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,0.530143,Braund,"Braund, Mr. Owen Harris",Mr,1912-04-10 12:00:00,s


2. **퀴즈 2:** **Embarked** 열의 값('S', 'C', 'Q')을 모두 소문자로 바꾸는 코드를 작성해보세요. 
    1. (대문자 'S','C','Q' → 소문자 's','c','q')

In [82]:
titanic_df['Embarked_low'] = titanic_df['Embarked'].str.lower()
titanic_df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fare_VS_ClassMean,LastName,Cleaned_name,Title,BoardTime,Embarked_low
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,0.530143,Braund,"Braund, Mr. Owen Harris",Braund,1912-04-10 12:00:00,s


## 시간 데이터 처리

> 문자열 날짜는 계산/추출이 어려움 → `pd.to_datetime()`으로 datetime으로 변환
> 
> 
> datetime 변환 후 `.dt`로 연/월/일/시 등 추출 가능
> 

- **시간(Date/Time) 데이터**는 문자열로 존재할 때 이를 **`datetime 타입`**으로 변환하여 다뤄야 한다.
    - 날짜/시간을 제대로 처리하면 **연도, 월, 일 단위로 묶어 분석**하거나 **두 시점의 차이를 계산**하는 등 시간에 특화된 처리를 쉽게 할 수 있습니다.
- 문자열 날짜 변환
    - **`pd.to_datetime()`** 함수를 사용 → 다양한 형식의 날짜 문자열을 자동으로 인식하여 `datetime64` 타입으로 변환
    - 형식을 지정해야 하는 경우 `format` 인자를 사용
    - detailed `format` 인자
        - 이 문자열 날짜가 어떤 모양인지를 **명시적으로 알려줘서** 그 형식대로 **빠르고(보통 더 빠름), 일관되게** `datetime64`로 파싱하게 만드는 옵션
        - `format`은 고정 문자 + 지시자(%로 시작)의 조합
            - `"2026-02-09"` → `"%Y-%m-%d"`
            - `"2026/02/09 13:45"` → `"%Y/%m/%d %H:%M"`
        - 지시자 종류
            - `%Y` : 4자리 연도 (예: 2026)
            - `%y` : 2자리 연도 (예: 26)
            - `%m` : 월(01~12)
            - `%d` : 일(01~31)
            - `%H` : 시(00~23)
            - `%I` : 시(01~12)
            - `%M` : 분(00~59)
            - `%S` : 초(00~59)
        
        ```python
        pd.to_datetime(s, format="%Y-%m-%d")               # 2026-02-09
        pd.to_datetime(s, format="%Y%m%d")                 # 20260209
        pd.to_datetime(s, format="%d/%m/%Y")               # 09/02/2026
        pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S")      # 2026-02-09 13:45:10
        ```
        
        - **혹시?**
            - `"DD/MM/YYYY"`처럼 **엑셀/사람식 표기**는 안 되고, 반드시 `"%d/%m/%Y"`처럼 **% 지시자**를 써야함.
            - 형식이 여러 개 섞여 있으면 `format` 하나로는 깨질 수 있다 → 이때는
                - 데이터 정제 후 통일하거나
                - `errors='coerce'`로 NaT 처리 후 예외만 따로 다루는 전략
- datetime으로 변환된 열(또는 Series)에 대해서는 **`.dt`** 접근자로 연월일시 등 **각각의 시간 구성 요소**를 쉽게 추출 가능
    - 예: `df['date'].dt.year`, `df['date'].dt.month`, `df['date'].dt.day`, `df['date'].dt.hour` 등 …
- 날짜 Series 간의 **뺄셈**을 하면 **시간 차이(`Timedelta`)**를 획득 가능
    - 예: `df['end_date'] - df['start_date']`를 하면 각 행마다 두 날짜의 차이가 `Timedelta` 형태로 계산

In [64]:
import pandas as pd

date_strings = pd.Series(['2025-01-01', '2025-03-15', '2025-03-31'])
print("문자열 형태 날짜:\n", date_strings)

문자열 형태 날짜:
 0    2025-01-01
1    2025-03-15
2    2025-03-31
dtype: object


In [ ]:
dates = pd.to_datetime(date_strings)
print("datetime 형태:\n", dates)
print("데이터 타입:", dates.dtype)
# 타입이 오브젝트에서 데이트 타임으로 바뀐 것을 확인할 수 있다.

datetime 형태:
 0   2025-01-01
1   2025-03-15
2   2025-03-31
dtype: datetime64[ns]
데이터 타입: datetime64[ns]


In [ ]:
print("연도:", dates.dt.year)
print("월:", dates.dt.month)
print("일:", dates.dt.day)
print("날짜:", dates.dt.date)
# 접근자로 .dt 사용

연도: 0    2025
1    2025
2    2025
dtype: int32
월: 0    1
1    3
2    3
dtype: int32
일: 0     1
1    15
2    31
dtype: int32
날짜: 0    2025-01-01
1    2025-03-15
2    2025-03-31
dtype: object


In [ ]:
diff = dates.iloc[2] - dates.iloc[0]
print("첫 번째와 마지막 날짜 차이:", diff)
print("차이 타입:", type(diff))
# 날짜 연산도 가능

첫 번째와 마지막 날짜 차이: 89 days 00:00:00
차이 타입: <class 'pandas._libs.tslibs.timedeltas.Timedelta'>


In [68]:
# Embarked 값에 따른 탑승 일시 매핑
embark_to_time = {'S': '1912-04-10 12:00:00',
                  'C': '1912-04-10 18:00:00',
                  'Q': '1912-04-11 12:00:00'}
# BoardTime 열 추가 (현재는 문자열 상태)
titanic_df['BoardTime'] = titanic_df['Embarked'].map(embark_to_time)
print(titanic_df[['Embarked', 'BoardTime']].head(3))

  Embarked            BoardTime
0        S  1912-04-10 12:00:00
1        C  1912-04-10 18:00:00
2        S  1912-04-10 12:00:00


- **예제 1: BoardTime 컬럼 datetime 변환** – 탑승 시간 컬럼인 BoardTime을  datetime 타입으로 변환

In [91]:
titanic_df['BoardTime'] = pd.to_datetime(titanic_df['BoardTime'])
titanic_df['BoardTime'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: BoardTime
Non-Null Count  Dtype         
--------------  -----         
889 non-null    datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 7.1 KB


- **예제 2: 기초적인 시간 연산 및 추출**
    1. **최초/마지막 탑승 시간 확인:** 모든 승객 중 가장 이른 탑승 시각과 가장 늦은 탑승 시각을 구해봅시다 (`min`, `max` 이용).
    2. **날짜별 탑승 인원 집계:** 날짜(일 단위-`dt.date`)별로 몇 명이 탑승했는지 계산해보겠습니다. (`value_counts()`)

In [94]:
print(titanic_df['BoardTime'].min())
print(titanic_df['BoardTime'].max())
titanic_df['BoardDate'] = titanic_df['BoardTime'].dt.date
titanic_df['BoardDate'].value_counts()

1912-04-10 12:00:00
1912-04-11 12:00:00


BoardDate
1912-04-10    812
1912-04-11     77
Name: count, dtype: int64

- **예제 3: 시간 간격 계산**
    - 각 승객이 **최초 탑승 시각(4월 10일 12:00)** 이후 얼마나 후에 탑승했는지 계산해보겠습니다.
    1. `BoardTime`에서 최초 탑승 시간을 빼서 `Timedelta`를 구해보겠습니다. (각 승객의 탑승 시각 - 최초 탑승 시각)
    2. 이를 시간(hour) 단위로도 변환해 보겠습니다. → 초 단위로 나와있는 `timedelta`를 3600으로 나눠 시간으로 변환(`dt.total_seconds()` → 초 단위로 바꿔줌)

In [102]:
titanic_df['TimeDelta'] = titanic_df['BoardTime'] - titanic_df['BoardTime'].min()
print(titanic_df['TimeDelta'])
titanic_df['HourTimeDelta'] = titanic_df['TimeDelta'].dt.total_seconds()/3600
print(titanic_df['HourTimeDelta'])

0     0 days 00:00:00
1     0 days 06:00:00
2     0 days 00:00:00
3     0 days 00:00:00
4     0 days 00:00:00
            ...      
886   0 days 00:00:00
887   0 days 00:00:00
888   0 days 00:00:00
889   0 days 06:00:00
890   1 days 00:00:00
Name: TimeDelta, Length: 891, dtype: timedelta64[ns]
0       0.0
1       6.0
2       0.0
3       0.0
4       0.0
       ... 
886     0.0
887     0.0
888     0.0
889     6.0
890    24.0
Name: HourTimeDelta, Length: 891, dtype: float64


퀴즈: 방금 생성한 BoardTime 열에서 시(hour) 정보를 추출하여 BoardHour라는 새로운 열을 추가해보세요. (힌트: datetime Series에 대해 dt.hour 속성을 사용할 수 있습니다. 예: df['datetime컬럼'].dt.hour)

In [69]:
import pandas as pd

# 학생 시험 점수 예제
exam_df1 = pd.DataFrame({'ID': [1, 2, 3],
                         '국어': [90, 85, 80]})
exam_df2 = pd.DataFrame({'ID': [2, 3, 4],
                         '영어': [75, 80, 85]})
print("exam_df1:\n", exam_df1)
print("exam_df2:\n", exam_df2)

exam_df1:
    ID  국어
0   1  90
1   2  85
2   3  80
exam_df2:
    ID  영어
0   2  75
1   3  80
2   4  85


In [70]:
merged_inner = pd.merge(exam_df1, exam_df2, on='ID', how='inner')
print("Inner merge 결과:\n", merged_inner)

Inner merge 결과:
    ID  국어  영어
0   2  85  75
1   3  80  80


In [71]:
merged_outer = pd.merge(exam_df1, exam_df2, on='ID', how='outer')
print("Outer merge 결과:\n", merged_outer)

Outer merge 결과:
    ID    국어    영어
0   1  90.0   NaN
1   2  85.0  75.0
2   3  80.0  80.0
3   4   NaN  85.0


In [72]:
classA = pd.DataFrame({'ID': [1, 2, 3],
                       'Score': [50, 60, 70]})
classB = pd.DataFrame({'ID': [4, 5],
                       'Score': [80, 90]})
combined = pd.concat([classA, classB], axis=0, ignore_index=True)
print("행 방향 결합 결과:\n", combined)

행 방향 결합 결과:
    ID  Score
0   1     50
1   2     60
2   3     70
3   4     80
4   5     90


In [73]:
names = pd.Series(['Alice', 'Bob', 'Charlie'])
scores = pd.Series([85, 92, 88])
merged_df = pd.concat([names, scores], axis=1)
merged_df.columns = ['Name', 'Score']
print("열 방향 결합 결과:\n", merged_df)

열 방향 결합 결과:
       Name  Score
0    Alice     85
1      Bob     92
2  Charlie     88


In [74]:
# 데이터프레임을 임의로 두 부분으로 분할
titanic_part1 = titanic_df.iloc[:445]   # 처음 445개 행
titanic_part2 = titanic_df.iloc[445:]   # 나머지 행

In [75]:
import pandas as pd

s = pd.Series([10, 20, 30])
print("원본 Series:\n", s)

원본 Series:
 0    10
1    20
2    30
dtype: int64


In [76]:
# 딕셔너리를 이용하여 값 매핑: 10->'A', 20->'B', 30->'C'
mapped = s.map({10: 'A', 20: 'B', 30: 'C'})
print("map 결과:\n", mapped)

map 결과:
 0    A
1    B
2    C
dtype: object


In [77]:
# 각 값에 5를 더하고 문자열로 변환하는 lambda 함수 적용
applied = s.apply(lambda x: str(x + 5))
print("apply 결과:\n", applied)
print("apply 결과 타입:", applied.dtype)

apply 결과:
 0    15
1    25
2    35
dtype: object
apply 결과 타입: object


In [78]:
df = pd.DataFrame({'A': [1,2,3], 'B': [10, 20, 30]})
print("원본 DF:\n", df)

# 각 행별 합계를 계산하여 시리즈로 반환 (axis=1)
row_sum = df.apply(lambda row: row['A'] + row['B'], axis=1)
print("각 행의 A+B 합:\n", row_sum)

# 이 결과를 새로운 열로 추가할 수도 있음
df['A+B'] = row_sum
print("새 열 추가 후 DF:\n", df)

원본 DF:
    A   B
0  1  10
1  2  20
2  3  30
각 행의 A+B 합:
 0    11
1    22
2    33
dtype: int64
새 열 추가 후 DF:
    A   B  A+B
0  1  10   11
1  2  20   22
2  3  30   33


In [79]:
df_idx = titanic_df.set_index("PassengerId")

In [80]:
df_back = df_idx.reset_index()

In [81]:
table = titanic_df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean")